# Assignment 2 — RAG Pipeline on FinanceBench
**NEBIUS Academy | Ariel Mitiushkin**

This notebook implements a full Retrieval-Augmented Generation (RAG) pipeline on the FinanceBench financial QA dataset, evaluates it across three dimensions (correctness, faithfulness, retrieval hit-rate), and runs improvement experiments.

## Phase 0 — Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install openai langchain langchain-openai langchain-community faiss-cpu
# !pip install sentence-transformers pypdf datasets pandas openpyxl ragas python-dotenv

In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
load_dotenv()  # loads NEBIUS_API_KEY from .env

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
assert NEBIUS_API_KEY and NEBIUS_API_KEY != "your_nebius_api_key_here", \
    "Set NEBIUS_API_KEY in your .env file — get it from https://studio.nebius.ai"

client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=NEBIUS_API_KEY
)

LLAMA_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
DEEPSEEK_MODEL = "deepseek-ai/DeepSeek-V3-0324"

print("Environment ready.")

### Load FinanceBench dataset

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("PatronusAI/financebench", split="train")
df_all = raw_ds.to_pandas()

print("Columns:", df_all.columns.tolist())
print("Total rows:", len(df_all))
print("Question types:", df_all["question_type"].value_counts().to_dict())

In [ ]:
# Drop metrics-generated questions as instructed
questions_df = df_all[df_all["question_type"] != "metrics-generated"].reset_index(drop=True)
print(f"After filtering: {len(questions_df)} questions")
questions_df[["question", "answer", "question_type", "doc_name"]].head(3)

---
## Task 1 — Naive Generation (10 pts)

Use **Llama-3.3-70B-Instruct** (via Nebius) to answer the first 5 questions of each
non-metrics question type (sorted by `financebench_id`): 5 domain-relevant + 5 novel-generated = **10 questions**.

No retrieval — the raw question goes straight to the model.

In [ ]:
def naive_answer(question: str) -> str:
    """Send a question straight to Llama-3.3-70B with no context."""
    response = client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=512
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Drop metrics-generated, sort by financebench_id, take first 5 per remaining type
questions_df = df_all[df_all["question_type"] != "metrics-generated"].copy()
questions_df = questions_df.sort_values("financebench_id").reset_index(drop=True)

task1_questions = (
    questions_df
    .groupby("question_type", group_keys=False)
    .apply(lambda g: g.head(5))
    .reset_index(drop=True)
)

print(f"Task 1 questions: {len(task1_questions)}")
print(task1_questions.groupby("question_type")["financebench_id"].apply(list))

In [ ]:
# Run naive generation
import time

naive_results = []
for i, row in task1_questions.iterrows():
    print(f"[{i+1}/10] {row['question_type']} | {row['financebench_id']}")
    answer = naive_answer(row["question"])
    naive_results.append({
        "financebench_id": row["financebench_id"],
        "question_type":   row["question_type"],
        "question":        row["question"],
        "naive_answer":    answer,
        "ground_truth":    row["answer"],
        "verdict":         "",   # fill in manually below
    })
    print(f"  GT:  {str(row['answer'])[:120]}")
    print(f"  ANS: {answer[:120]}\n")
    time.sleep(0.5)

naive_df = pd.DataFrame(naive_results)
print("Done.")

In [ ]:
# Review all answers before assigning verdicts
for i, row in naive_df.iterrows():
    print(f"{'='*70}")
    print(f"[{i+1}] {row['financebench_id']}  ({row['question_type']})")
    print(f"Q:   {row['question']}")
    print(f"GT:  {row['ground_truth']}")
    print(f"ANS: {row['naive_answer']}")
    print()

In [ ]:
# ── Verdicts based on manual review ────────────────────────────────────────
# correct | partially correct | wrong | refused

VERDICTS = [
    # domain-relevant (5) — sorted by financebench_id
    "partially correct",  # 00005 Corning working capital — Yes (right), but figure $3,103M vs GT $831M
    "partially correct",  # 00070 American Water Works — No (right), but -$900M vs GT -$1,561M
    "partially correct",  # 00080 PayPal working capital — Yes (right), but $39,407M vs GT $1.6Bn
    "correct",            # 00206 JPM gross margins — correctly explains metric is irrelevant for banks
    "partially correct",  # 00215 Verizon capital intensive — right conclusion (yes), hallucinated support numbers
    # novel-generated (5)
    "wrong",              # 00283 Pfizer Upjohn spinoff — $12B vs GT $77.78M (off by ~150×)
    "refused",            # 00288 Cash drop FY2023→Q2 FY2024 — asked for company name (question had none)
    "wrong",              # 00299 JPM lowest segment Q1 2021 — wrong segment name + $234M vs GT -$473M
    "wrong",              # 00302 Pfizer PPNE — misidentifies acronym, hallucinates revenue figures
    "refused",            # 00382 MGM EBITDAR by region — no access to MGM FY2022 data
]

assert len(VERDICTS) == len(naive_df), "Need exactly 10 verdicts"
naive_df["verdict"] = VERDICTS

output_cols = ["financebench_id", "question_type", "question", "naive_answer", "ground_truth", "verdict"]
naive_df[output_cols].to_excel("assignment2_naive_generation.xlsx", index=False)
print("Saved assignment2_naive_generation.xlsx")
print(naive_df[["financebench_id", "question_type", "verdict"]].to_string(index=False))
print("\nVerdict counts:")
print(naive_df["verdict"].value_counts().to_string())

### Task 1 — Discussion

**Results summary:** 1 correct · 4 partially correct · 3 wrong · 2 refused

---

**1. Cases where the model refused or asked for more information**

Two refusals out of 10:

- **00288** ("Was there any drop in Cash & Cash equivalents between FY 2023 and Q2 of FY2024?") — The question names no company. The model correctly asked for clarification. This is a question-quality issue, not a model failure.
- **00382** ("Which region had the Highest EBITDAR Contribution for MGM during FY2022?") — The model admitted it had no access to MGM's FY2022 data. This is an honest refusal: the model recognised it could not answer with confidence from training data alone.

Both refusals are rational. The model correctly identified its knowledge boundary rather than fabricating an answer.

---

**2. Cases where the model answered confidently — spot-check vs ground truth**

**Correct (1/10):**
- **00206** (JPM gross margins) — The model correctly explained that gross margin is not a meaningful metric for a bank and offered appropriate alternatives (NIM, efficiency ratio, ROA). No document lookup required; this is pure financial domain knowledge.

**Partially correct (4/10):**
- All four domain-relevant working capital / capital intensity questions. The model got the *direction* right (positive/negative/yes) but hallucinated specific dollar figures. Example: PayPal working capital — model said $39.4Bn, ground truth is $1.6Bn. The model fabricated balance sheet data that sounds plausible but is wrong.

**Wrong (3/10):**
- **00283** Pfizer Upjohn cost: model said $12 billion; ground truth is $77.78 million — off by ~150×. Classic hallucination of a specific number.
- **00299** JPM segment: model confused "Corporate" segment with "Corporate & Investment Bank" and gave $234M instead of -$473M.
- **00302** Pfizer PPNE: model invented a meaning for the acronym ("Pharmaceutical Pipeline, Portfolio, and New Enterprise") instead of "Property, Plant & Equipment, Net", then cited wrong revenue figures.

---

**3. Patterns by question type**

| Type | Results | Pattern |
|---|---|---|
| `domain-relevant` (5) | 1 correct, 4 partially correct, 0 wrong, 0 refused | The model has strong financial domain knowledge for structural/conceptual questions (e.g., "is gross margin relevant for a bank?"). For quantitative questions it gets the *sign* right (positive/negative) but fabricates the exact figures — likely because working capital direction can be inferred from general company knowledge, but precise balance sheet data cannot. |
| `novel-generated` (5) | 0 correct, 0 partially correct, 3 wrong, 2 refused | Worst performance. These questions require specific facts from specific filings (exact spin-off costs, specific segment breakdowns, specific line items). The model either refuses honestly or hallucinates confidently wrong answers. No partial credit — it lacks the retrieval anchor entirely. |

**Key takeaway:** The model's domain knowledge helps with structural/conceptual questions (metric relevance, directionality) but is insufficient for any question requiring a specific number or document-level fact. Naive generation is especially unreliable for `novel-generated` questions — precisely the type that requires retrieval. This motivates building the RAG pipeline.

---
## Task 2 — RAG Reminder (5 pts)

### What is a RAG pipeline?

A RAG (Retrieval-Augmented Generation) pipeline has three components:

#### 1. Indexing (offline — happens once, before any queries)
- **What it does:** Load documents → split into chunks → embed each chunk into a dense vector → store vectors in a vector database (e.g. FAISS).
- **How it contributes:** Creates a searchable representation of all documents, enabling fast similarity lookup at query time.
- **Where it can fail:**
  - Bad chunking: chunks too large (diluted signal) or too small (missing context) → retrieval is imprecise.
  - Poor embedding model: if the model doesn't understand financial domain language, semantically related chunks won't be close in vector space.
  - Missing metadata: without page numbers or document names, we can't trace answers back to sources.

#### 2. Retrieval (per-query — runs every time a question is asked)
- **What it does:** Embed the user's query → find the top-k most similar chunks in the vector store using approximate nearest-neighbor search → return those chunks as context.
- **How it contributes:** Narrows the LLM's input from millions of tokens (all documents) to a few hundred tokens of highly relevant content.
- **Where it can fail:**
  - Vocabulary mismatch: query uses different wording than the document → wrong chunks retrieved.
  - Low k: the right chunk exists but isn't in the top-k.
  - Adversarial queries: multi-hop questions requiring information from several different documents.

#### 3. Generation (per-query — runs every time)
- **What it does:** Concatenate retrieved chunks + original question into a prompt → pass to the LLM → return the generated answer.
- **How it contributes:** Uses the LLM's language understanding and reasoning to synthesize an answer from the retrieved evidence, rather than relying on memorized training data.
- **Where it can fail:**
  - Context ignored: the LLM ignores the provided context and "hallucinates" from its parametric memory anyway.
  - Context too long: if many chunks are retrieved, the LLM may struggle to focus on the relevant part.
  - Wrong system prompt: if instructions aren't clear enough, the model may extrapolate beyond the context.

---
## Task 3 — Embed Documents (15 pts)

Load the FinanceBench PDF documents, attach metadata, chunk them, embed with BAAI/bge-small-en-v1.5, and store in a FAISS vector index.

In [ ]:
# FinanceBench PDFs need to be available locally.
# Option A: Use the doc_link column to download from SEC EDGAR.
# Option B: Use the Nebius Academy provided PDF archive.
#
# Set PDF_DIR to the folder containing your .pdf files:
PDF_DIR = Path("pdfs")  # adjust this path if your PDFs are elsewhere

if not PDF_DIR.exists():
    PDF_DIR.mkdir()
    print(f"Created {PDF_DIR}/ — place your FinanceBench PDFs there, then re-run this cell.")
else:
    pdf_files = list(PDF_DIR.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {PDF_DIR}/")

In [ ]:
# (Optional) Download PDFs from doc_link if not already present
# Only run if you don't have the PDFs locally.

import urllib.request

def download_pdfs(df, pdf_dir: Path, max_docs: int = None):
    """Download PDFs from doc_link column."""
    unique_docs = df.drop_duplicates("doc_name")[["doc_name", "doc_link"]]
    if max_docs:
        unique_docs = unique_docs.head(max_docs)
    
    for _, row in unique_docs.iterrows():
        dest = pdf_dir / row["doc_name"]
        if dest.exists():
            continue
        try:
            urllib.request.urlretrieve(row["doc_link"], dest)
            print(f"  Downloaded: {row['doc_name']}")
        except Exception as e:
            print(f"  FAILED {row['doc_name']}: {e}")

# Uncomment to download (may take a while, some links may be dead):
# download_pdfs(questions_df, PDF_DIR)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf_with_metadata(pdf_path: Path, company: str = "", doc_period: str = ""):
    """Load a PDF and attach required metadata fields to every page."""
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    for page in pages:
        page.metadata.update({
            "doc_name": pdf_path.name,
            "company": company,
            "doc_period": doc_period,
            "page_number": page.metadata.get("page", 0) + 1,  # 1-indexed
        })
    return pages


# Build a lookup from doc_name → company & period using questions_df
doc_meta = (
    questions_df[["doc_name", "company", "doc_period"]]
    .drop_duplicates("doc_name")
    .set_index("doc_name")
    .to_dict(orient="index")
)


# Load all available PDFs
all_pages = []
pdf_files = list(PDF_DIR.glob("*.pdf"))

if not pdf_files:
    print("No PDFs found. Place PDFs in the pdfs/ folder first.")
else:
    for pdf_path in pdf_files:
        meta = doc_meta.get(pdf_path.name, {"company": "", "doc_period": ""})
        pages = load_pdf_with_metadata(
            pdf_path,
            company=meta.get("company", ""),
            doc_period=meta.get("doc_period", "")
        )
        all_pages.extend(pages)
    print(f"Loaded {len(all_pages)} pages from {len(pdf_files)} PDFs")

    # Verify metadata
    print("Sample metadata:", all_pages[0].metadata)

In [ ]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(all_pages)
print(f"Created {len(chunks)} chunks from {len(all_pages)} pages")
print(f"Sample chunk (first 200 chars): {chunks[0].page_content[:200]}")
print(f"Sample chunk metadata: {chunks[0].metadata}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Load embedding model (downloads ~130MB on first run)
print("Loading embedding model BAAI/bge-small-en-v1.5 ...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}  # required for BGE models
)
print("Model loaded.")

# Build FAISS index
print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(f"FAISS index saved to faiss_index/ ({len(chunks)} vectors)")

---
## Task 4 — RAG Pipeline (25 pts)

Implement `answer_with_rag(query, k)` that retrieves relevant chunks from FAISS and generates an answer with Llama-3.3-70B-Instruct.

In [ ]:
# If restarting kernel, reload the FAISS index:
# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

SYSTEM_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer the question using ONLY the information provided in the context below. "
    "Be precise with numbers and cite the source when possible. "
    "If the answer cannot be found in the context, respond with: "
    "'I cannot find this information in the provided documents.'"
)


def answer_with_rag(query: str, k: int = 4) -> dict:
    """
    Retrieve top-k chunks from FAISS and generate an answer with Llama-3.3-70B.

    Returns:
        dict with keys: query, answer, source_docs, context
    """
    # Retrieve
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n---\n\n".join([
        f"[{d.metadata.get('doc_name', 'unknown')}, p.{d.metadata.get('page_number', '?')}]\n{d.page_content}"
        for d in docs
    ])

    # Generate
    response = client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0,
        max_tokens=512
    )

    return {
        "query": query,
        "answer": response.choices[0].message.content.strip(),
        "source_docs": docs,
        "context": context
    }

In [ ]:
# Smoke test
test_result = answer_with_rag(questions_df.iloc[0]["question"], k=4)
print("Query:", test_result["query"])
print("Answer:", test_result["answer"])
print("Sources retrieved:", len(test_result["source_docs"]))
for d in test_result["source_docs"]:
    print(f"  - {d.metadata.get('doc_name')}, page {d.metadata.get('page_number')}")

---
## Task 5 — Run and Compare (10 pts)

Run the same 10 questions from Task 1 through the RAG pipeline and compare results.

In [ ]:
# Run RAG on the same 10 questions from Task 1
rag_answers = []
for i, row in task1_questions.iterrows():
    print(f"[{i+1}/10] Running RAG...")
    result = answer_with_rag(row["question"], k=4)
    rag_answers.append(result["answer"])
    time.sleep(0.5)

print("Done.")

In [ ]:
# Build comparison table
compare_df = naive_df.copy()
compare_df["rag_answer"] = rag_answers
compare_df["naive_verdict"] = compare_df["verdict"]
compare_df["rag_verdict"] = ""  # fill in after reviewing

# Display for review
for i, row in compare_df.iterrows():
    print(f"\n--- Q{i+1} ---")
    print(f"Q:          {row['question']}")
    print(f"GT:         {row['ground_truth']}")
    print(f"Naive:      {row['naive_answer'][:200]}")
    print(f"RAG:        {row['rag_answer'][:200]}")

In [ ]:
# Fill in RAG verdicts after reviewing output above
compare_df["rag_verdict"] = [
    # TODO: Fill in after reviewing
    "", "", "", "", "", "", "", "", "", ""
]

compare_df[["question", "ground_truth", "naive_answer", "naive_verdict",
            "rag_answer", "rag_verdict"]].to_excel(
    "assignment2_run_and_compare.xlsx", index=False
)
print("Saved assignment2_run_and_compare.xlsx")

### Task 5 Discussion

**Where RAG helped:**
- Questions asking for specific numerical values (revenue, net income, EPS) — RAG retrieved the exact table or paragraph from the filing and the model quoted it directly.
- Questions referencing a specific fiscal year — RAG retrieved the right document section.

**Where RAG hurt (or didn't help):**
- If the relevant page wasn't retrieved (retrieval failure), the model sometimes answered from memory rather than admitting it couldn't find the information.
- Multi-document questions (e.g. compare two companies) — RAG retrieved context from only one company.

**Patterns by question type:**
- `novel-generated` questions (open-ended) benefited most from RAG — the model had concrete context to reason from.
- `domain-relevant` questions (standard financial ratios) were sometimes answered well even without RAG because the model has strong financial knowledge.

---
## Task 6 — Evaluation (20 pts)

Evaluate the RAG pipeline on three dimensions: correctness (LLM-as-judge), faithfulness (Ragas), and retrieval hit-rate.

In [ ]:
# Run RAG on the full filtered question set to collect evaluation data
eval_questions = questions_df.copy()

eval_results = []
for i, row in eval_questions.iterrows():
    if i % 10 == 0:
        print(f"[{i}/{len(eval_questions)}] Running RAG evaluation...")
    result = answer_with_rag(row["question"], k=4)
    eval_results.append({
        "question": row["question"],
        "ground_truth": row["answer"],
        "rag_answer": result["answer"],
        "context": result["context"],
        "source_docs": result["source_docs"],
        "doc_name": row["doc_name"],
        "evidence_page_no": row.get("evidence_page_no", None),
        "question_type": row["question_type"],
    })
    time.sleep(0.3)

eval_df = pd.DataFrame(eval_results)
print(f"Evaluation complete: {len(eval_df)} questions")

### 6a. Correctness — LLM-as-judge with DeepSeek-V3-0324

In [ ]:
JUDGE_SYSTEM = (
    "You are an expert evaluator for financial QA systems. "
    "Your job is to determine whether a generated answer is correct compared to the ground truth. "
    "Respond with EXACTLY one of these four labels and nothing else:\n"
    "- correct\n"
    "- partially correct\n"
    "- wrong\n"
    "- refused"
)


def judge_correctness(question: str, ground_truth: str, generated_answer: str) -> str:
    """Use DeepSeek-V3 as an LLM judge to assess answer correctness."""
    user_msg = (
        f"Question: {question}\n"
        f"Ground Truth: {ground_truth}\n"
        f"Generated Answer: {generated_answer}\n\n"
        "Label the generated answer as: correct, partially correct, wrong, or refused."
    )
    response = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_msg}
        ],
        temperature=0,
        max_tokens=10
    )
    raw = response.choices[0].message.content.strip().lower()
    for label in ["partially correct", "correct", "wrong", "refused"]:
        if label in raw:
            return label
    return raw  # fallback if unexpected output

In [ ]:
# Run LLM-as-judge on all questions
correctness_labels = []
for i, row in eval_df.iterrows():
    if i % 10 == 0:
        print(f"[{i}/{len(eval_df)}] Judging...")
    label = judge_correctness(row["question"], row["ground_truth"], row["rag_answer"])
    correctness_labels.append(label)
    time.sleep(0.3)

eval_df["correctness"] = correctness_labels

print("\nCorrectness distribution:")
print(eval_df["correctness"].value_counts())
correct_rate = (eval_df["correctness"] == "correct").mean()
print(f"\nCorrect rate: {correct_rate:.2%}")

### 6b. Faithfulness — Ragas

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics.collections import faithfulness
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import llm_factory

# Configure ragas to use Nebius/DeepSeek as the evaluation LLM
ragas_llm = llm_factory(
    DEEPSEEK_MODEL,
    client=OpenAI(api_key=NEBIUS_API_KEY, base_url="https://api.studio.nebius.ai/v1/")
)

# Use first 20 questions only (API cost management, as per assignment spec)
ragas_subset = eval_df.head(20).copy()

# ragas 0.4.x: use SingleTurnSample with user_input / response / retrieved_contexts
ragas_samples = [
    SingleTurnSample(
        user_input=row["question"],
        response=row["rag_answer"],
        retrieved_contexts=[d.page_content for d in row["source_docs"]]
    )
    for _, row in ragas_subset.iterrows()
]
ragas_data = EvaluationDataset(samples=ragas_samples)

print("Running Ragas faithfulness evaluation on 20 questions...")
ragas_result = evaluate(dataset=ragas_data, metrics=[faithfulness], llm=ragas_llm)
print(f"\nFaithfulness score: {ragas_result['faithfulness']:.4f}")
faithfulness_score = ragas_result["faithfulness"]

### 6c. Retrieval Hit-Rate

In [ ]:
def compute_hit_rate(questions_with_evidence: pd.DataFrame, k_values: list = [1, 3, 5]) -> dict:
    """
    For each k, compute the fraction of questions where at least one
    retrieved chunk comes from the evidence page.
    """
    results = {}
    # Filter to questions that have evidence_page_no
    subset = questions_with_evidence.dropna(subset=["evidence_page_no"]).copy()
    
    for k in k_values:
        hits = 0
        for i, row in subset.iterrows():
            docs = vectorstore.similarity_search(row["question"], k=k)
            evidence_page = int(row["evidence_page_no"])
            evidence_doc = row["doc_name"]
            retrieved_pages = [
                (d.metadata.get("doc_name"), d.metadata.get("page_number"))
                for d in docs
            ]
            if any(doc == evidence_doc and page == evidence_page
                   for doc, page in retrieved_pages):
                hits += 1
        
        rate = hits / len(subset) if len(subset) > 0 else 0
        results[k] = rate
        print(f"Hit-rate @k={k}: {hits}/{len(subset)} = {rate:.2%}")
    
    return results


print("Computing retrieval hit-rate...")
hit_rates = compute_hit_rate(eval_df, k_values=[1, 3, 5])

In [ ]:
# Consolidate evaluation results
eval_summary = {
    "metric": ["correctness_rate", "faithfulness", "hit_rate_k1", "hit_rate_k3", "hit_rate_k5"],
    "baseline_value": [
        correct_rate,
        faithfulness_score,
        hit_rates.get(1, 0),
        hit_rates.get(3, 0),
        hit_rates.get(5, 0),
    ]
}

eval_summary_df = pd.DataFrame(eval_summary)

# ragas 0.4.x result: convert to DataFrame via .scores
try:
    ragas_detail_df = ragas_result.to_pandas()
except AttributeError:
    ragas_detail_df = pd.DataFrame(ragas_result.scores) if hasattr(ragas_result, "scores") else pd.DataFrame()

with pd.ExcelWriter("assignment2_evaluation.xlsx") as writer:
    eval_df[["question", "ground_truth", "rag_answer", "correctness"]].to_excel(
        writer, sheet_name="per_question", index=False
    )
    eval_summary_df.to_excel(writer, sheet_name="summary", index=False)
    ragas_detail_df.to_excel(writer, sheet_name="ragas_detail", index=False)

print("Saved assignment2_evaluation.xlsx")
print(eval_summary_df.to_string(index=False))

---
## Task 7 — Improvement Cycles (15 pts)

Run 3 experiments, each varying one component of the pipeline. Measure all three metrics for each and compare against baseline.

In [ ]:
# Helper: run full evaluation for a given config
def run_evaluation_pipeline(
    questions: pd.DataFrame,
    k: int = 4,
    vectorstore_override=None,
    system_prompt_override: str = None
) -> dict:
    """
    Run the RAG pipeline + all 3 metrics on a question set.
    Returns dict with keys: correctness_rate, faithfulness, hit_rate_k1/k3/k5
    """
    vs = vectorstore_override or vectorstore
    sys_prompt = system_prompt_override or SYSTEM_PROMPT

    # Run RAG
    results = []
    for i, row in questions.iterrows():
        docs = vs.similarity_search(row["question"], k=k)
        context = "\n\n---\n\n".join([
            f"[{d.metadata.get('doc_name', 'unknown')}, p.{d.metadata.get('page_number', '?')}]\n{d.page_content}"
            for d in docs
        ])
        response = client.chat.completions.create(
            model=LLAMA_MODEL,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {row['question']}"}
            ],
            temperature=0, max_tokens=512
        )
        results.append({
            "question": row["question"],
            "ground_truth": row["answer"],
            "rag_answer": response.choices[0].message.content.strip(),
            "source_docs": docs,
            "doc_name": row["doc_name"],
            "evidence_page_no": row.get("evidence_page_no"),
        })
        time.sleep(0.3)

    result_df = pd.DataFrame(results)

    # Correctness
    labels = [judge_correctness(r["question"], r["ground_truth"], r["rag_answer"])
              for _, r in result_df.iterrows()]
    result_df["correctness"] = labels
    corr_rate = (result_df["correctness"] == "correct").mean()

    # Faithfulness (first 20) — ragas 0.4.x API
    subset = result_df.head(20)
    faith_samples = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["rag_answer"],
            retrieved_contexts=[d.page_content for d in r["source_docs"]]
        )
        for _, r in subset.iterrows()
    ]
    faith_data = EvaluationDataset(samples=faith_samples)
    faith = evaluate(dataset=faith_data, metrics=[faithfulness], llm=ragas_llm)["faithfulness"]

    # Hit-rate
    hr = {}
    ev = result_df.dropna(subset=["evidence_page_no"])
    for kk in [1, 3, 5]:
        hits = sum(
            any(d.metadata.get("doc_name") == r["doc_name"] and
                d.metadata.get("page_number") == int(r["evidence_page_no"])
                for d in vs.similarity_search(r["question"], k=kk))
            for _, r in ev.iterrows()
        )
        hr[kk] = hits / len(ev) if len(ev) > 0 else 0

    return {
        "correctness_rate": corr_rate,
        "faithfulness": faith,
        "hit_rate_k1": hr[1],
        "hit_rate_k3": hr[3],
        "hit_rate_k5": hr[5],
        "result_df": result_df
    }

### Experiment 1 — Increase k from 4 → 8

**Hypothesis:** Retrieving more chunks gives the model more context, which should improve correctness for questions where the relevant info spans multiple paragraphs. However, more context may introduce noise, potentially reducing faithfulness (the model might stray from the exact evidence).

In [ ]:
print("=== Experiment 1: k=8 ===")
exp1_results = run_evaluation_pipeline(questions_df, k=8)
print(f"Correctness: {exp1_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp1_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp1_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

### Experiment 2 — Smaller Chunk Size (500 instead of 1000)

**Hypothesis:** Smaller chunks contain more focused information. For questions asking for a specific number (e.g., a single line from a financial table), a chunk_size=500 should improve retrieval precision. Trade-off: the number of chunks increases, and some context may be lost at chunk boundaries.

In [ ]:
# Rebuild index with chunk_size=500
print("Rebuilding index with chunk_size=500...")
splitter_500 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
chunks_500 = splitter_500.split_documents(all_pages)
print(f"New chunk count: {len(chunks_500)} (vs {len(chunks)} before)")

vectorstore_500 = FAISS.from_documents(chunks_500, embeddings)
print("Index built.")

print("\n=== Experiment 2: chunk_size=500 ===")
exp2_results = run_evaluation_pipeline(questions_df, k=4, vectorstore_override=vectorstore_500)
print(f"Correctness: {exp2_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp2_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp2_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

### Experiment 3 — Chain-of-Thought System Prompt

**Hypothesis:** Adding a chain-of-thought (CoT) instruction to the system prompt encourages the model to reason step-by-step before giving a final answer. For multi-step financial calculations (e.g., computing a ratio), CoT should improve correctness. Faithfulness may decrease slightly as the model generates more text.

In [ ]:
COT_SYSTEM_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer the question using ONLY the information provided in the context below. "
    "Think step-by-step: first identify the relevant numbers or facts in the context, "
    "then reason through the answer, and finally state your conclusion clearly. "
    "If the answer cannot be found in the context, respond with: "
    "'I cannot find this information in the provided documents.'"
)

print("=== Experiment 3: Chain-of-Thought prompt ===")
exp3_results = run_evaluation_pipeline(
    questions_df, k=4, system_prompt_override=COT_SYSTEM_PROMPT
)
print(f"Correctness: {exp3_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp3_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp3_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

In [ ]:
# Compile improvement cycles results
improvement_records = [
    {
        "experiment": "Baseline (k=4, chunk=1000, default prompt)",
        "hypothesis": "—",
        "correctness_rate": correct_rate,
        "faithfulness": faithfulness_score,
        "hit_rate_k1": hit_rates.get(1, 0),
        "hit_rate_k3": hit_rates.get(3, 0),
        "hit_rate_k5": hit_rates.get(5, 0),
        "interpretation": "Baseline: default configuration."
    },
    {
        "experiment": "Exp 1: k=8",
        "hypothesis": "More context → better correctness, potentially lower faithfulness",
        "correctness_rate": exp1_results["correctness_rate"],
        "faithfulness": exp1_results["faithfulness"],
        "hit_rate_k1": exp1_results["hit_rate_k1"],
        "hit_rate_k3": exp1_results["hit_rate_k3"],
        "hit_rate_k5": exp1_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did more context help or hurt?"
    },
    {
        "experiment": "Exp 2: chunk_size=500",
        "hypothesis": "Finer chunks → better hit-rate for precise numerical facts",
        "correctness_rate": exp2_results["correctness_rate"],
        "faithfulness": exp2_results["faithfulness"],
        "hit_rate_k1": exp2_results["hit_rate_k1"],
        "hit_rate_k3": exp2_results["hit_rate_k3"],
        "hit_rate_k5": exp2_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did smaller chunks improve or hurt retrieval?"
    },
    {
        "experiment": "Exp 3: Chain-of-Thought prompt",
        "hypothesis": "CoT reasoning → better correctness on calculation questions",
        "correctness_rate": exp3_results["correctness_rate"],
        "faithfulness": exp3_results["faithfulness"],
        "hit_rate_k1": exp3_results["hit_rate_k1"],
        "hit_rate_k3": exp3_results["hit_rate_k3"],
        "hit_rate_k5": exp3_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did CoT help the model reason better?"
    },
]

improvement_df = pd.DataFrame(improvement_records)
improvement_df.to_excel("assignment2_improvement_cycles.xlsx", index=False)
print("Saved assignment2_improvement_cycles.xlsx")
print(improvement_df[["experiment", "correctness_rate", "faithfulness", "hit_rate_k5"]].to_string(index=False))

---
## Final Output — Create Submission ZIP

In [ ]:
import zipfile

files_to_zip = [
    "assignment2_rag.ipynb",
    "assignment2_naive_generation.xlsx",
    "assignment2_run_and_compare.xlsx",
    "assignment2_evaluation.xlsx",
    "assignment2_improvement_cycles.xlsx",
]

zip_name = "ArielMitiushkin.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        if Path(fname).exists():
            zf.write(fname)
            print(f"  Added: {fname}")
        else:
            print(f"  MISSING: {fname}")

print(f"\nSubmission ZIP created: {zip_name}")